# Mamba: TaskTracker train -> PasteTrace test

Bat GPU T4 truoc khi chay. Pipeline nay co dinh vai tro du lieu:

- `tasktracker/`: train va validation.
- `pastetrace/`: external test, khong fit scaler va khong update model.

## 0a. Kiem tra GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Hay bat GPU T4 truoc khi train Mamba')

## 0b. Cai dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'scikit-learn', 'ninja', 'packaging'], check=True)
if importlib.util.find_spec('mamba_ssm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-build-isolation', 'causal-conv1d', 'mamba-ssm'], check=True)
from mamba_ssm import Mamba
print('Dependencies OK')

## 0c. Clone/pull project

In [ ]:
import os

REPO_URL = 'https://github.com/lequocviet-3103/Fraud-Detection.git'
REPO_DIR = '/content/Fraud-Detection'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## 1. Kiem tra va build hai bo du lieu

In [ ]:
for folder in ['tasktracker', 'pastetrace']:
    labels = os.path.join(folder, 'normalized', 'labels.csv')
    if not os.path.isfile(labels):
        raise FileNotFoundError(labels)
    print('OK:', labels)

!python -m src.data.build_sequences --train-dir tasktracker --test-dir pastetrace --min-events 1
!python -m src.data.make_splits --val 0.15 --seed 42

import pandas as pd
train_df = pd.read_csv('data/mamba/train_index.csv')
test_df = pd.read_csv('data/mamba/test_index.csv')
print('TaskTracker:', len(train_df), train_df.label.value_counts().to_dict())
print('PasteTrace :', len(test_df), test_df.label.value_counts().to_dict())

## 2. Train tren TaskTracker

Lenh duoi dung `--train-all`: scaler va model dung toan bo TaskTracker; khong co validation/early stopping.

In [ ]:
!python -m src.models.mamba_model train \
    --d-model 64 --n-layers 2 --dropout 0.2 \
    --epochs 80 --lr 1e-3 --patience 10 \
    --batch-size 8 --max-len 1000 --train-all

## 3. External test tren PasteTrace

Chi chay khi da chot hyperparameter. Lenh nay chi load model/scaler da dong bang.

In [ ]:
!python -m src.models.mamba_model test

import json
with open('results/mamba_metrics.json', encoding='utf8') as handle:
    results = json.load(handle)
results['mamba']

## 4. Luu model va ket qua

In [ ]:
import zipfile

artifacts = [
    'models/mamba/mamba.pt',
    'models/mamba/config.json',
    'models/mamba/scaler.json',
    'results/mamba_metrics.json',
    'results/mamba_predictions.csv',
]
with zipfile.ZipFile('/content/mamba_tasktracker_to_pastetrace.zip', 'w') as archive:
    for path in artifacts:
        if os.path.isfile(path):
            archive.write(path)
print('Saved /content/mamba_tasktracker_to_pastetrace.zip')

## 5. Predict mot JSON normalized (tuy chon)

In [ ]:
SESSION_JSON = 'pastetrace/normalized/111_A.json'
!python -m src.models.mamba_model predict {SESSION_JSON}